In [ ]:
import os
import sys
import json
import re

import numpy as np
import pandas as pd
from collections import defaultdict

import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize

# Parent directory PATH
sys.path.append("..")

from utils.json import load_json, save_json
from utils.dataset import json_keys, json_structure, extract_documents, extract_annotations, remove_punctuation, remove_punctuation_dataframe, extract_annotations_and_split_documents

In [ ]:
# Load the training data
file = "training.json"
path = "../data/raw/"
data = load_json(os.path.join(path, file))

In [ ]:
print(f"Type of loaded data: {type(data)}")
print(f"Number of documents: {len(data)}")

In [ ]:
# Explore your JSON data
json_structure(data, max_depth=10)

In [ ]:
# Get the JSON keys structure
structure = json_keys(data)

display(structure)

In [ ]:
# Insights using the first document
record_first = data[0]

if isinstance(data, list):
    print(f"Type document: {type(data[0])}")
    print(f"Keys document: {structure['main_keys']}")
    print()

    for main_keys in record_first:
        print(f"Key: {main_keys}")
        print(f"Type: {type(record_first[main_keys])}")
        print(f"Length: {len(record_first[main_keys])}")
        print(f"Value: {record_first[main_keys]}")
        print()

        if "data" in main_keys:
            for k in record_first["data"].keys():
                print(f"\tKey: {k}")
                print(f"\tType: {type(record_first['data'][k])}")
                print(f"\tLength: {len(record_first['data'][k])}")
                print(f"\tValue: {record_first['data'][k]}")
                print()

In [ ]:
extract_documents(data)

In [ ]:
df_train = extract_annotations_and_split_documents(data)

In [ ]:
print(f"Extracted {len(df_train)} annotations")
display(df_train)

In [ ]:
annotation_counts = df_train.groupby("doc_id").size().sort_values(ascending=False)

print("TOP documents by annotation count:")
for doc_id, count in annotation_counts.head(10).items():
    print(f"Document {doc_id} has {count} annotations")

In [ ]:
# Duplicated documents
true_counts = {}

# Count each unique document ID only once
for doc in data:
    doc_id = doc["data"]["id"]
    if doc_id in true_counts:
        true_counts[doc_id] += 1
    else:
        true_counts[doc_id] = 1

# Find actually duplicated IDs
actual_duplicates = {id: count for id, count in true_counts.items() if count > 1}

print(f"Number of documents: {len(data)}")
print(f"Number of unique document IDs: {len(true_counts)}")
print(f"Number of truly duplicated IDs: {len(actual_duplicates)}")
print()

if actual_duplicates:
    print("True duplicate document IDs:")
    for doc_id, count in actual_duplicates.items():
        print(f"ID {doc_id} appears {count} times")
else:
    print("No duplicate document IDs found")

In [ ]:
# Count the occurrences of each label
label_counts = df_train["label"].value_counts()

print("Label Distribution:")
for label, count in label_counts.items():
    print(f"{label}: {count} ({count/len(df_train)*100:.2f}%)")

labels = label_counts.index
counts = label_counts.values

# Plot the distribution using Matplotlib directly
plt.figure(figsize=(12, 7))
plt.bar(labels, counts, alpha=0.75)
plt.title("Distribution of Label Types")
plt.xlabel("Count")
plt.ylabel("Label")
plt.tight_layout()
plt.show()

In [ ]:
# we remove the punctuation from the scopes and add a new column to the dataframe with it
df_train = remove_punctuation_dataframe(df_train)

In [ ]:
# Common Negation Cues
neg_cues = df_train[df_train["label"] == "NEG"]["clean_text"].value_counts()
print("TOP Negation Cues: \n")
for cue, count in neg_cues.items():
    print(f"{cue}: {count}")

In [ ]:
# Common Uncertainty Cues
unc_cues = df_train[df_train["label"] == "UNC"]["clean_text"].value_counts()
print("TOP Uncertainty Cues: \n")
for cue, count in unc_cues.items():
    print(f"{cue}: {count}")

In [ ]:
# Calculate scope lengths
df_train["text_length"] = df_train["text"].apply(len)

scope_labels = ["NSCO", "USCO"]

# Average Scope's 
for label in scope_labels:
    avg_len = df_train[df_train["label"] == label]["text_length"].mean()
    print(f"Average {label} length: {avg_len:.2f} characters")

# Plot scope length distributions by label type
plt.figure(figsize=(12, 7))
for label in scope_labels:
    scope_lengths = df_train[df_train["label"] == label]["text_length"]
    plt.hist(scope_lengths, bins=50, alpha=0.75, label=label)

plt.title("Distribution of Scope Lengths")
plt.xlabel("Length in Characters")
plt.ylabel("Frequency")
plt.legend()
plt.xlim(0, 100) # Limit to most common
plt.tight_layout()
plt.show()

### Group Scope with Cue

We can see the labels that are in each sentence and verify any outliers or error. If we see a clear pattern in these errors, we can change our line parser (the function that determines what is and what is not a punctuation period).

In [ ]:
def group_labels(df):
    # create a new column to store the scope for a CUE
    df["scope"] = None

    # group the Context and Cues that are consecutive
    list_rows = [row for row in df['line_number'].unique()]
    for row in list_rows:
        scope_count  = 0
        df_row = df[df['line_number'] == row]
        print(row)
        values_labels = df_row['label'].values
        print(values_labels)

        # outliers
        if row == '20103430_3': # this is very weird case that seems to be a mistake on the labeling
            continue

        match len(df_row):
            case 0:
                pass
            case 1:
                if df_row['label'].values[0] == 'NEG':
                    df.loc[df_row.index[0], 'scope'] = f"{row}_{scope_count}"
                elif df_row['label'].values[0] == 'UNC':
                    df.loc[df_row.index[0], 'scope'] = f"{row}_{scope_count}"
                else:
                    raise ValueError(f"Single label {df_row['label'].values[0]} in row {row}\ntext: {df_row['text'].values[0]}")
            case 2:
                if 'NEG' in values_labels and 'NSCO' in values_labels:
                    for index in df_row.index:
                        df.loc[index, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                elif 'UNC' in values_labels and 'USCO' in values_labels:
                    for index in df_row.index:
                        df.loc[index, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                elif 'NEG' in values_labels and 'UNC' in values_labels:
                    df.loc[0, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                    df.loc[1, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                elif values_labels[0] == 'NEG' and values_labels[1] == 'NEG':
                    df.loc[0, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                    df.loc[1, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                elif values_labels[0] == 'UNC' and values_labels[1] == 'UNC':
                    df.loc[0, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                    df.loc[1, 'scope'] = f"{row}_{scope_count}"
                    scope_count += 1
                else:
                    raise ValueError(f"Two labels {values_labels} in row {row}\ntext: {df_row['text'].values[0]}")


    return df

In [ ]:
grouped_df = group_labels(df_train)
#grouped_df = transform_scope(grouped_df)

# TODO - Unknow error occurs